# mart_dev.runs_per_day

One row per job and day: how many runs that job had that day, for 2026-09-23 and 2026-09-24,
counted from ops.job_runs. Partitioned by dt.

Made by `project.new_table("mart_dev.runs_per_day")` from `admin/templates/saved_table.py`.
Its logic is written by hand: the code cells run top to bottom once its Table reference is
written from the logic.

The code cells: 1 the Project and your query; 2 the logic, which starts `# logic`; 3 try it;
4 its lineage; 5 its Hive, in mart_dev__runs_per_day.hql; 6 its rows in the Reference sheet.

In [ ]:
# 1. The Project and your query
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project folder: two up from here

from sqlglot_composer import export_hive, export_lineage, project, run_query, show_hive
from warehouse import query

In [ ]:
# logic
# 2. The logic: imports, the rows it reads, and functions only
from sqlglot_composer import (
    AS, FROM, GROUP_BY, INSERT_OVERWRITE, SELECT, WHERE, Clause, Statement, between,
    count_rows, create_table, statement,
)
from references.mart_dev__runs_per_day import runs_per_day  # this table's own Table reference
from references.ops__job_runs import job_runs  # the table it reads

# The rows write() reads: one job's runs for each day from 2026-09-23 to 2026-09-24, both in.
# ops.job_runs is partitioned by dt, so the read filters dt.
READS = between(job_runs.dt, "2026-09-23", "2026-09-24")

# This table's partition columns, in the order its SELECT gives them, last: [] for none.
PARTITIONED_BY = ["dt"]


def create() -> Statement:
    """Step 1: create mart_dev.runs_per_day from its Table reference, if it isn't there yet."""
    return create_table(runs_per_day, may_exist=True)


def rows() -> list[Clause]:
    """The clauses preview and write share, so the two can never disagree: the runs of each
    job on each day, counted from the rows READS picks.

    The partition column comes last in SELECT, and stays in GROUP_BY, so each day's rows
    stay apart.
    """
    return [
        SELECT(
            job_runs.job_id,  # what each row is for
            AS(count_rows(), "runs"),
            job_runs.dt,  # this table's partition column, last
        ),
        FROM(job_runs),
        WHERE(READS),
        GROUP_BY(
            job_runs.job_id,
            job_runs.dt,  # kept, so each Partition's rows stay apart
        ),
    ]


def preview() -> Statement:
    """Step 2: the rows write() would save, as a SELECT to run and check first."""
    return statement(*rows())


def write() -> Statement:
    """Step 3: save the rows, replacing the Partitions they fill: sent twice, it leaves the
    same rows, not twice as many."""
    return statement(INSERT_OVERWRITE(runs_per_day), *rows())

In [ ]:
# 3. Try it: the Hive of each step, then the rows
show_hive([create(), write()])
run_query(preview(), query)

In [ ]:
# 4. Lineage: a new pair in lineage/, named by the time and this table, beside the ones before
export_lineage(write())

In [ ]:
# 5. Its Hive: mart_dev__runs_per_day.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# 6. The Reference sheet: this table's rows, from references/mart_dev__runs_per_day.py
project.refresh_reference_sheet("mart_dev.runs_per_day")